In [1]:
import numpy as np
from scipy.spatial.transform import Rotation as Rotation
import yaml
import open3d as o3d
from pathlib import Path

ImportError: KeyboardInterrupt: 

At:
  /usr/local/lib/python3.8/dist-packages/numpy/core/arrayprint.py(1463): _array_repr_implementation
  <frozen importlib._bootstrap>(219): _call_with_frames_removed
  <frozen importlib._bootstrap_external>(1166): create_module
  <frozen importlib._bootstrap>(556): module_from_spec
  <frozen importlib._bootstrap>(657): _load_unlocked
  <frozen importlib._bootstrap>(975): _find_and_load_unlocked
  <frozen importlib._bootstrap>(991): _find_and_load
  /home/tony/.local/lib/python3.8/site-packages/open3d/__init__.py(75): <module>
  <frozen importlib._bootstrap>(219): _call_with_frames_removed
  <frozen importlib._bootstrap_external>(848): exec_module
  <frozen importlib._bootstrap>(671): _load_unlocked
  <frozen importlib._bootstrap>(975): _find_and_load_unlocked
  <frozen importlib._bootstrap>(991): _find_and_load
  /tmp/ipykernel_52862/1325388942.py(4): <module>
  /home/tony/.local/lib/python3.8/site-packages/IPython/core/interactiveshell.py(3508): run

In [ ]:
def seven_num2matrix(translation,roatation):#translation x,y,z rotation x,y,z,w
    transform_matrix = np.identity(4)
    transform_matrix[:3,:3] = Rotation.from_quat(roatation).as_matrix()
    transform_matrix[:3,3] = translation
    return transform_matrix

object_tracking_tf_path = "/media/tony/新加卷/camera_data/banana/pose_result/result.txt"
quat_translation = np.loadtxt(object_tracking_tf_path)


In [ ]:
matrix = []
for index in np.arange(quat_translation.shape[0]):
    matrix.append(seven_num2matrix(quat_translation[index][4:],quat_translation[index][[1,2,3,0]]))

matrix = np.array(matrix)

In [ ]:
rgb_transform = np.array([
        [
            0.3221832568741056,
            0.57349689219082,
            -0.7531927134787391,
            1.9653506143136346
        ],
        [
            0.9353371981154199,
            -0.315619076454712,
            0.15977773436706177,
            -0.114133351627669
        ],
        [
            -0.14608995451977763,
            -0.7559668731004995,
            -0.6381001582534389,
            1.3065810986484299
        ],
        [
            0.0,
            0.0,
            0.0,
            1.0
        ]
    ])

In [ ]:
result = np.einsum('lj,ijk->ilk',rgb_transform,matrix)

def transform_mesh_with_matrix(transform_matrix,mesh):
    vertices = np.asarray(mesh.vertices).copy() 
    vertices = np.hstack((vertices,np.ones((vertices.shape[0],1))),dtype=float).T
    transformed_vertices = np.dot(transform_matrix[:3,:],vertices).T
    mesh.vertices = o3d.utility.Vector3dVector(transformed_vertices)
    return mesh


In [ ]:
model_path = "/media/tony/新加卷/camera_data/banana/models/banana.obj"

result_folder = "/media/tony/新加卷/camera_data/banana/object_result_position/"

for index in np.arange(matrix.shape[0]):
    mesh = o3d.io.read_triangle_mesh(model_path)
    mesh = transform_mesh_with_matrix(result[index],mesh)
    mesh.simplify_quadric_decimation(int((0.01) * len(mesh.triangles)))
    o3d.io.write_triangle_mesh(str(Path(result_folder) / Path(str(index)+".ply")),mesh)

